In [1]:
import os
# Read environment variables
base_url = os.getenv("OPENAI_BASE_URL")
api_key = os.getenv("OPENAI_API_KEY")

In [13]:
from openai import OpenAI  

client = OpenAI()  

response = client.responses.create( 
    model="openai.gpt-oss-120b", 
    input=[ 
        {"role": "user", "content": "Write a one-sentence bedtime story about a unicorn."} 
    ] 
)  

print(response.output_text)

As the moon rose, the silver‑mane unicorn gently tucked the sleepy stars into a cloud‑soft blanket, humming a lullaby that drifted through the night and whisked every dreaming child into the sweetest, glitter‑filled sleep.


In [18]:
import os
import json
import requests
from openai import OpenAI

client = OpenAI()

# 1. Define the actual Python search function
def perform_web_search(query: str) -> str:
    """Executes search using Tavily API (or any web search API)."""
    api_key = os.getenv("TAVILY_API_KEY")
    response = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": api_key, "query": query, "max_results": 3}
    )
    return json.dumps(response.json())

# 2. Define the tool schema using type="function"
tools = [
    {
        "type": "function",
        "function": {
            "name": "perform_web_search",
            "description": "Searches the live web for current real-time information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query string to look up."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# 3. Call the model with the valid tool schema
response = client.chat.completions.create(
    model="openai.gpt-oss-120b",  # or your preferred model
    messages=[
        {"role": "user", "content": "What is the latest news about quantum computing today?"}
    ],
    tools=tools,
    tool_choice="auto"
)

# 4. Check if the model requested a search call
tool_calls = response.choices[0].message.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        if tool_call.function.name == "perform_web_search":
            args = json.loads(tool_call.function.arguments)
            search_results = perform_web_search(args["query"])
            print("Search Results Retrieved:", search_results)

Search Results Retrieved: {"query": "quantum computing news September 29 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.nasdaq.com/press-release/01-quantum-showcase-quantum-safe-encrypted-ai-analytics-gcxpo-2026-2026-09-22", "title": "01 Quantum to Showcase Quantum-Safe Encrypted AI ...", "content": "7 days ago \u2014 Hosted by Invest Ottawa and Area X.O, the event will take place on September 29, 2026, from 8:00 a.m. to 4:00 p.m. EDT at Area X.O, 1740\u00a0...Read more", "score": 0.80735475, "raw_content": null, "id": "4f83c5-00"}, {"url": "https://thequantuminsider.com/2026/07/29/israel-pushes-for-national-quantum-rd-center-to-unite-industry", "title": "Israel Pushes For National Quantum R&D Center to Unite Industry", "content": "The Quantum Insider (TQI) is the leading source for the latest quantum technology news, market intelligence, industry analysis, and data covering the global quantum ecosystem.\n\nNEWSROOM\n\nPLATFORM\n\nSER